## model training using EfficientNetB0

import libraries and settings

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import matplotlib.pyplot as plt

In [2]:
# image resize dimensions
image_size = (224, 224)

# batch size
batch_size = 32

# dataset directory (update if needed)
dataset_path = "../dataset_8"

In [3]:
train_data_generator = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.3,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2]
)

validation_data_generator = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.3
)

create training & validation generator

In [4]:
training_generator = train_data_generator.flow_from_directory(
    dataset_path,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42  # for reproducibility
)

validation_generator = validation_data_generator.flow_from_directory(
    dataset_path,
    target_size=image_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42  # for reproducibility
)

FileNotFoundError: [WinError 3] The system cannot find the path specified: '../dataset_8'

In [ ]:
print(f"Training samples: {training_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")
print(f"Number of classes: {training_generator.num_classes}")
print(f"Class indices: {training_generator.class_indices}")

In [ ]:
images, labels = next(training_generator)

plt.figure(figsize=(10, 10))
for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(images[i])
    plt.title(f"Label: {np.argmax(labels[i])}")
    plt.axis('off')
plt.show()

using EfficientNetB0

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# load EfficientNetB0 base model without top layers
base_model = EfficientNetB0(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False  # freeze base model for initial training

# build full model
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dense(4, activation='softmax')  # 4 output classes for dataset
])

# compile model
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

In [ ]:
early_stop = EarlyStopping(patience=5, restore_best_weights=True)

history = model.fit(
    training_generator,
    epochs=20,
    validation_data=validation_generator,
    callbacks=[early_stop]
)

evaluate training performance

In [ ]:
# plot accuracy
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# plot loss
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()


### observations

even after 5-10 epochs, if validation accuracy is stuck at 25%, that's a clear sign of overfitting or poor generalization.

fine tune base model

In [ ]:
base_model.trainable = True

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

early_stop = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)

history_fine = model.fit(
    training_generator,
    epochs=20,
    validation_data=validation_generator,
    callbacks=[early_stop]
)


In [ ]:
print(training_generator.class_indices)
print(validation_generator.class_indices)

In [ ]:
val_loss, val_acc = model.evaluate(validation_generator)
print(f"Validation Accuracy: {val_acc*100:.2f}%")

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Predictions
Y_pred = model.predict(validation_generator)
y_pred = np.argmax(Y_pred, axis=1)

# True labels
y_true = validation_generator.classes

# Class labels in correct order
class_names = list(validation_generator.class_indices.keys())

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.show()


### observations

- model only predicts "glioma"
    - model has been overfitted/ collapsed to a single-class output
    - dataset may be too small, only 336 training images

what to do next:
- stronger data augmentation
- fine-tune base models
- consider simpler models
